In [ ]:
# [셀 1] 패키지 설치 및 환경 변수 설정
!pip install -qU langchain langchain-openai langchain-core langgraph

import os
import getpass

# 50불 크레딧이 있는 학생들의 개인 API 키를 입력받습니다.
print("OpenAI API 키를 입력하세요:")
os.environ["OPENAI_API_KEY"] = getpass.getpass()

In [ ]:
# [셀 2] Tool Calling의 기본 원리 이해하기
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

print("=== 1. 도구(Tool) 정의하기 ===")
# tool 데코레이터와 Docstring(설명)이 핵심입니다. LLM은 이 설명을 보고 도구를 고릅니다.
@tool
def calculate_tax(income: int) -> float:
    """주어진 소득(income)에 대해 10%의 세금을 계산하여 반환합니다."""
    return income * 0.1

tools = [calculate_tax]

# gpt-4o-mini 모델에 도구를 '바인딩(연결)' 합니다.
llm_with_tools = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

# LLM이 도구를 사용한 것인지, 스스로 답변한 것인지 판별해주는 헬퍼 함수
def analyze_response(response):
    # 1. tool_calls 리스트에 데이터가 들어있다면? -> 도구를 쓰기로 결심한 것!
    if response.tool_calls:
        print("🛠️ LLM의 판단: '도구가 필요합니다!'")
        print(f"  - 선택한 도구 이름: {response.tool_calls[0]['name']}")
        print(f"  - 도구에 넣을 인자(값): {response.tool_calls[0]['args']}")
        print(f"  - 텍스트 내용(content)은 비어있나요?: {response.content == ''}")

    # 2. tool_calls가 비어있고 content에 내용이 있다면? -> 스스로 답변한 것!
    elif response.content:
        print("💬 LLM의 판단: '도구 없이 스스로 답변할 수 있습니다!'")
        print(f"  - LLM의 텍스트 답변: {response.content}")
        print(f"  - 도구 호출 내역(tool_calls)은 비어있나요?: {len(response.tool_calls) == 0}")
    print("-" * 50)


print("\n=== 2. LLM의 응답 확인하기 ===")

# 테스트 1: 도구가 필요한 질문
question_1 = "내 월급이 3000000원인데, 세금이 얼마야?"
print(f"[질문 1] {question_1}")
response_1 = llm_with_tools.invoke(question_1)
analyze_response(response_1)

# 테스트 2: 도구가 필요 없는 일반적인 질문
question_2 = "안녕, 넌 누구니?"
print(f"\n[질문 2] {question_2}")
response_2 = llm_with_tools.invoke(question_2)
analyze_response(response_2)

In [ ]:
# [셀 3] LangGraph를 활용한 최신 ReAct 에이전트 구축
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 1. 두 가지 다른 역할을 하는 도구 준비
@tool
def get_word_length(word: str) -> int:
    """단어의 길이를 알파벳/글자 수로 반환합니다."""
    return len(word)

@tool
def multiply(a: int, b: int) -> int:
    """두 숫자를 곱합니다."""
    return a * b

tools = [get_word_length, multiply]

# 2. 모델 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. LangGraph 기반 최신 에이전트 생성
agent_executor = create_react_agent(llm, tools)

print("=== 에이전트 실행 테스트 ===")
question = "'Supercalifragilisticexpialidocious'라는 단어의 길이를 구하고, 그 길이에 5를 곱해줘."
print(f"질문: {question}\n")

# LangGraph는 입력값을 딕셔너리 안의 "messages" 리스트 형태로 받습니다.
result = agent_executor.invoke({"messages": [("user", question)]})

# result["messages"]에는 질문, 도구 사용 내역, 최종 답변이 모두 기록되어 있습니다.
# 전체 로그를 확인하고 싶다면 print(result["messages"])
print(f"최종 결과: {result['messages'][-1].content}")

In [ ]:
# [셀 4] RAG를 도구로 활용하는 최신 Agentic RAG
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 1. 가상의 RAG 데이터베이스 구축
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings)

vectorstore.add_texts([
    "우리 회사의 핵심 가치는 '끊임없는 도전'과 '상호 존중'입니다.",
    "2026년도 전사 워크샵은 10월 15일 제주도에서 진행될 예정입니다."
])
retriever = vectorstore.as_retriever()
print("RAG 벡터 저장소 구축 완료!\n")

# 2. Retriever를 도구로 변환
@tool
def search_company_database(query: str) -> str:
    """회사의 핵심 가치, 워크샵 일정, 사내 정책에 대해 물어볼 때 이 도구를 사용하여 검색하세요."""
    docs = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

tools = [search_company_database]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. 시스템 프롬프트를 prompt 인자에 전달 (state_modifier -> prompt 로 변경!)
system_prompt = "당신은 사내 안내 봇입니다. 필요하다면 search_company_database 도구를 사용하세요."

agent_executor = create_react_agent(llm, tools, prompt=system_prompt)

print("=== Agentic RAG 테스트 ===")
# 케이스 A: 도구를 사용해야 하는 질문
print("\n[테스트 1] 워크샵 일정 질문")
result_a = agent_executor.invoke({"messages": [("user", "올해 전사 워크샵 언제, 어디서 해?")]})
print(result_a["messages"][-1].content)

# 케이스 B: 도구를 사용하지 않아도 되는 질문
print("\n[테스트 2] 일반적인 인사")
result_b = agent_executor.invoke({"messages": [("user", "안녕? 오늘 날씨 진짜 좋다!")]})
print(result_b["messages"][-1].content)

In [ ]:
# [셀 5] 다중 도구(Multi-Tool)를 자유자재로 다루는 에이전트
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 1. 서로 다른 성격의 도구 3가지 정의
@tool
def weather_search(city: str) -> str:
    """특정 도시의 현재 날씨를 반환합니다."""
    # 실습용 더미 데이터
    weather_data = {"서울": "맑음, 25도", "도쿄": "비, 20도", "뉴욕": "흐림, 15도"}
    return weather_data.get(city, "해당 지역의 날씨 정보가 없습니다.")

@tool
def calculate_exchange_rate(usd_amount: float) -> float:
    """달러(USD)를 원화(KRW)로 환전한 금액을 계산합니다. (1달러 = 1350원 기준)"""
    return usd_amount * 1350

@tool
def extract_city_name(text: str) -> str:
    """사용자의 문장에서 도시 이름만 정확히 추출합니다."""
    # LLM이 텍스트 처리를 위해 호출할 수 있는 도구
    return f"추출된 도시: {text}"

tools = [weather_search, calculate_exchange_rate, extract_city_name]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. 에이전트 생성
agent_executor = create_react_agent(llm, tools)

print("=== 다중 도구 라우팅 테스트 ===")
# 복합 질문: 날씨 조회 + 환율 계산이 모두 필요한 상황
complex_question = "나 다음주에 뉴욕 가는데, 거기 날씨 어때? 그리고 여행 경비로 500달러 가져가려는데 한국 돈으로 얼마야?"
print(f"질문: {complex_question}\n")

result = agent_executor.invoke({"messages": [("user", complex_question)]})

# 학생들에게 에이전트가 어떤 순서로 도구를 호출했는지 로그를 살펴보게 합니다.
for msg in result["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        print(f"🛠️ LLM의 도구 선택: {[t['name'] for t in msg.tool_calls]}")

print(f"\n✅ 최종 답변: {result['messages'][-1].content}")

In [ ]:
# [셀 6] 코드를 작성하고 실행하는 코딩 에이전트 구축
import sys
import io
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
# 최신 LangChain V1.0 표준에 맞춘 에이전트 생성 함수 임포트
from langchain.agents import create_agent

# 1. Experimental 패키지 대신, 파이썬 내장 기능을 활용한 커스텀 REPL 도구 직접 제작
@tool
def python_repl(code: str) -> str:
    """파이썬 코드를 실행하고 print()로 출력된 결과값을 반환합니다."""
    # 표준 출력(stdout)을 가로채서 LLM이 볼 수 있는 문자열로 저장하기 위한 설정
    old_stdout = sys.stdout
    new_stdout = io.StringIO()
    sys.stdout = new_stdout

    try:
        # LLM이 작성한 코드를 실제 파이썬 환경에서 실행
        # (주의: 실제 상용 서비스에서는 보안상 매우 위험하므로 샌드박스 환경이 필요함을 인지해야 합니다!)
        exec(code, {})
        result = new_stdout.getvalue()
    except Exception as e:
        result = f"코드 실행 중 에러 발생: {e}"
    finally:
        sys.stdout = old_stdout # 원래의 표준 출력으로 원복

    return result if result.strip() else "코드 실행이 완료되었으나 출력값이 없습니다."

tools = [python_repl]

# 2. 시스템 프롬프트 설정
system_prompt_text = """당신은 뛰어난 파이썬 데이터 분석가입니다.
질문을 해결하기 위해 python_repl 도구를 사용하여 코드를 작성하고 실행하세요.
반드시 print() 함수를 사용하여 결과를 출력해야만 관찰(Observation)할 수 있습니다."""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. 최신 create_agent 사용 (state_modifier -> system_prompt 로 변경)
agent_executor = create_agent(llm, tools, system_prompt=system_prompt_text)

print("=== 코딩 에이전트 테스트 ===")
coding_question = "1부터 1000까지의 숫자 중에서 7의 배수이면서 13의 배수인 숫자들의 리스트를 구하고, 그 총합을 계산해서 알려줘."
print(f"질문: {coding_question}\n")

# LangGraph 기반은 messages 딕셔너리로 입력을 전달합니다.
result = agent_executor.invoke({"messages": [("user", coding_question)]})

# 에이전트가 호출한 도구 내역과 실제 실행한 코드 확인
for msg in result["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        print("💻 LLM이 작성한 파이썬 코드:\n")
        # LLM이 전달한 인자(args)의 키 값이 'code'인지 확인 (tool 정의에 따라 달라짐)
        print(msg.tool_calls[0]['args'].get('code', msg.tool_calls[0]['args']))
        print("-" * 30)

print(f"\n✅ 최종 답변: {result['messages'][-1].content}")

In [ ]:
# [셀 7] 이전 대화를 기억하는 상태 유지형(Stateful) 에이전트
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver # LangGraph의 핵심 메모리 모듈

@tool
def get_user_info(name: str) -> str:
    """사용자의 사내 정보를 조회합니다."""
    db = {"홍길동": "개발팀 팀장", "김철수": "디자인팀 사원"}
    return db.get(name, "정보 없음")

tools = [get_user_info]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. 메모리 세이버 객체 생성
memory = MemorySaver()

# 2. checkpointer 인자에 메모리 객체를 연결하여 에이전트 생성
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

# 3. 세션 ID 설정 (누구와의 대화인지 구분하는 고유 키)
config = {"configurable": {"thread_id": "student_session_01"}}

print("=== 메모리 에이전트 대화 테스트 ===")
# 턴 1: 정보 제공 및 도구 사용
print("[사용자] 내 이름은 홍길동이야. 내 직급이 뭔지 사내 망에서 찾아줄래?")
res1 = agent_executor.invoke(
    {"messages": [("user", "내 이름은 홍길동이야. 내 직급이 뭔지 사내 망에서 찾아줄래?")]},
    config=config
)
print(f"[AI 답변] {res1['messages'][-1].content}\n")

# 턴 2: 명시적인 정보 없이 대명사 사용 (기억력 테스트)
print("[사용자] 그럼 내가 속한 팀은 어디야?")
res2 = agent_executor.invoke(
    {"messages": [("user", "그럼 내가 속한 팀은 어디야?")]},
    config=config # 동일한 thread_id를 넘겨주면 이전 대화를 기억합니다!
)
print(f"[AI 답변] {res2['messages'][-1].content}")

In [ ]:
# [셀 8] 승인 없이는 실행되지 않는 안전한 에이전트
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

@tool
def send_email(to: str, content: str) -> str:
    """지정된 주소로 이메일을 전송합니다. (매우 민감한 도구)"""
    return f"[{to}]에게 이메일 전송 완료: {content}"

tools = [send_email]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
memory = MemorySaver()

# interrupt_before 옵션을 사용하여 특정 노드(여기서는 'tools' 노드) 실행 직전에 멈추게 합니다.
agent_executor = create_react_agent(
    llm,
    tools,
    checkpointer=memory,
    interrupt_before=["tools"] # 도구가 호출되기 직전에 무조건 일시 정지!
)

config = {"configurable": {"thread_id": "approval_test_01"}}

print("=== 1단계: 사용자 명령 및 일시 정지 ===")
user_input = "ceo@nextai.com으로 '연봉 인상 바랍니다'라고 메일 좀 당장 보내줘."
print(f"사용자: {user_input}\n")

# 스트리밍 모드로 실행하여 멈추는 시점을 확인합니다.
for event in agent_executor.stream({"messages": [("user", user_input)]}, config):
    print("실행 중 이벤트:", list(event.keys()))

# 상태를 확인해보면 다음 실행할 곳(next)이 'tools'로 되어 있고 대기 중입니다.
state = agent_executor.get_state(config)
print(f"\n🚨 [경고] 에이전트가 도구 실행을 대기 중입니다. 다음 실행 노드: {state.next}")

print("\n=== 2단계: 관리자 승인 후 계속 실행 ===")
user_approval = input("관리자님, 에이전트가 이메일을 보내려고 합니다. 승인하시겠습니까? (y/n): ")

if user_approval.lower() == 'y':
    print("승인 완료. 도구 실행을 재개합니다...")
    # None을 입력으로 주어 멈췄던 지점부터 다시 실행(Resume) 시킵니다.
    for event in agent_executor.stream(None, config):
        pass

    final_state = agent_executor.get_state(config)
    print(f"\n✅ 최종 결과: {final_state.values['messages'][-1].content}")
else:
    print("관리자가 실행을 거부했습니다.")